Import PDF Document :

In [3]:
import os 
import requests

pdf_path = "knowledge_base/artificial_intelligence_technology.pdf"

if not os.path.exists(pdf_path):
  print(f"[Info] file doesn't exist, downloading... ")

  #Enter the url of the pdf 
  url = "https://link.springer.com/content/pdf/10.1007/978-981-19-2879-6.pdf"

  # Local filename to save the downloaded file 
  file_name = pdf_path

  # Send a GET request to the url 
  response = requests.get(url , timeout=5)

  # check if the request was successful
  if response.status_code == 200:
    # Open the file and save it 
    with open (file_name,"wb") as  file:
      file.write(response.content)
    print(f"[Info] the file has been downloaded and saved as {file_name}")

  else:
    print(f"[Info] Failed to download the file. Status code: {response.status_code}")
else:
  print(f"File {pdf_path} Exists.")


File knowledge_base/artificial_intelligence_technology.pdf Exists.


Ingestion / Open the pdf and extract the text and metadata from the pdf:

In [10]:
import fitz
from tqdm.auto import tqdm

def text_format(text:str) -> str:
  """Performs minor formatting on text."""
  clear_text = text.replace("\n", " ").strip()
  # Other potential text formatting functions can go here
  return clear_text

# Open PDF and get lines/pages
# Note: this only focuses on text, rather than images/figures etc
def open_and_read_pdf(pdf_path : str)-> list[dict]:
  """
    Opens a PDF file, reads its text content page by page, and collects statistics.

    Parameters:
        pdf_path (str): The file path to the PDF document to be opened and read.

    Returns:
        list[dict]: A list of dictionaries, each containing the page number
        (adjusted), character count, word count, sentence count, token count, and the extracted text
        for each page.
    """
  doc = fitz.open(pdf_path) # Open a document
  pages_and_texts = []
  for page_number,page in tqdm(enumerate(doc,start=1)):
    text = page.get_text()
    text = text_format(text)
    if not text:
      continue

    # pages_and_texts.append({"page_number": page_number - 13,  # adjust page numbers since our PDF starts on page 14
    #                             "page_char_count": len(text),
    #                             "page_word_count": len(text.split(" ")),
    #                             "page_sentence_count_raw": len(text.split(". ")),
    #                             "page_token_count": len(text) / 4,  # 1 token = ~4 chars,
    #                             "text": text})
    pages_and_texts.append({"text": text,
                            "metadata":{
                              "source": pdf_path,
                              "pdf_page": page_number,
                              "page_label": page.get_label()
                            }})
  return pages_and_texts
pages_and_texts = open_and_read_pdf("knowledge_base/artificial_intelligence_technology.pdf")
print(len(pages_and_texts))
print(pages_and_texts[0])
print(pages_and_texts[13])
print(pages_and_texts[-1])


0it [00:00, ?it/s]

308it [00:01, 230.88it/s]

308
{'text': 'Official Textbooks for Huawei ICT Academy ARTIFICIAL  INTELLIGENCE  TECHNOLOGY Huawei Technologies Co., Ltd.', 'metadata': {'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 1, 'page_label': 'C1'}}
{'text': 'Chapter 1 A General Introduction to Artiﬁcial Intelligence The emergence and rise of artiﬁcial intelligence undoubtedly played an important role during the development of the Internet. Over the past decade, with extensive applications in the society, artiﬁcial intelligence has become more relevant to people’s daily life. This chapter introduces the concept of artiﬁcial intelligence, the related technologies, and the existing controversies over the topic. 1.1 The Concept of Artiﬁcial Intelligence 1.1.1 What Is Artiﬁcial Intelligence? Currently, people mainly learn about artiﬁcial intelligence (AI) through news, movies, and the applications in daily life, as shown by Fig. 1.1. A rather widely accepted deﬁnition of AI, also a relatively early

In [5]:
doc = fitz.open("knowledge_base/artificial_intelligence_technology.pdf")

print("Total PDF pages:",len(doc))
print("Extracted pages:",len(pages_and_texts))

Total PDF pages: 308
Extracted pages: 308


In [6]:
print(pages_and_texts[0]["metadata"])
print(pages_and_texts[12]["metadata"])
print(pages_and_texts[13]["metadata"])
print(pages_and_texts[-1]["metadata"])

{'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 1, 'page_label': 'C1'}
{'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 13, 'page_label': 'xiii'}
{'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 14, 'page_label': '1'}
{'source': 'knowledge_base/artificial_intelligence_technology.pdf', 'pdf_page': 308, 'page_label': '298'}


Get some stats on the text

In [7]:
import pandas as pd 

df = pd.DataFrame(pages_and_texts)
df.head()

,text,metadata
0,Official Textbooks for Huawei ICT Academy ARTI...,{'source': 'knowledge_base/artificial_intellig...
1,Artiﬁcial Intelligence Technology,{'source': 'knowledge_base/artificial_intellig...
2,"Huawei Technologies Co., Ltd. Artiﬁcial Intell...",{'source': 'knowledge_base/artificial_intellig...
3,"Huawei Technologies Co., Ltd. Hangzhou, China ...",{'source': 'knowledge_base/artificial_intellig...
4,Preface The rapid development of information t...,{'source': 'knowledge_base/artificial_intellig...


In [8]:
df.describe().round(2)

,text,metadata
count,308,308
unique,308,308
top,Official Textbooks for Huawei ICT Academy ARTI...,{'source': 'knowledge_base/artificial_intellig...
freq,1,1
